# Demonstração Guiada: PostgreSQL/PostGIS no Google Colab

**Objetivo:** mostrar, de maneira panorâmica, como um banco geográfico pode ser
consultado com SQL e utilizado pelo Python para gerar tabelas e mapas.

> Os alunos não precisam memorizar os comandos de instalação e configuração.
> As primeiras etapas podem ser executadas pelo professor.

O banco utilizado contém **dados fictícios** e geometrias simplificadas.


## Visão geral do fluxo

1. Instalar PostgreSQL, PostGIS e bibliotecas Python;
2. Criar um banco temporário no Colab;
3. Enviar e importar o arquivo `.sql`;
4. Criar uma conexão;
5. Fazer consultas simples;
6. Gerar mapas;
7. Executar uma consulta espacial.


## 1. Preparar o ambiente — professor executa

Esta célula instala os programas necessários na máquina temporária do Colab.
Quando a sessão for reiniciada, esta etapa precisará ser executada novamente.


In [ ]:
%%capture
!sudo apt-get update -qq
!sudo apt-get install -y postgresql postgresql-contrib postgis
!pip install -q pandas geopandas sqlalchemy psycopg2-binary matplotlib folium mapclassify


In [ ]:
!sudo service postgresql start
print("PostgreSQL iniciado.")


 * Starting PostgreSQL 14 database server
   ...done.
PostgreSQL iniciado.


## 2. Criar o banco temporário — professor executa

A senha abaixo é apenas didática e vale somente dentro da sessão temporária.
Não utilize essa senha em um servidor real.


In [ ]:
import subprocess

DB_NAME = "geoprocessamento_demo"
DB_USER = "postgres"
DB_PASSWORD = "123"

comandos = [
    [
        "sudo", "-u", "postgres", "psql", "-c",
        f"ALTER USER postgres WITH PASSWORD '{DB_PASSWORD}';"
    ],
    ["sudo", "-u", "postgres", "dropdb", "--if-exists", DB_NAME],
    ["sudo", "-u", "postgres", "createdb", DB_NAME],
]

for comando in comandos:
    subprocess.run(comando, check=True)

print(f"Banco temporário criado: {DB_NAME}")


Banco temporário criado: geoprocessamento_demo


## 3. Enviar o banco de teste

Execute a célula e selecione o arquivo:

`Banco_PostGIS_Teste_Manaus.sql`


In [ ]:
# pegar o arquivo do computador
from google.colab import files
from pathlib import Path

enviados = files.upload()

arquivos_sql = [
    nome for nome in enviados
    if nome.lower().endswith(".sql")
]

if not arquivos_sql:
    raise ValueError("Nenhum arquivo .sql foi enviado.")

arquivo_sql = Path("/content") / arquivos_sql[0]
print("Arquivo selecionado:", arquivo_sql)


In [ ]:
# Puxar o arquivo diretamente do git
!wget -O /content/saneamento_restricoes_vs14.sql https://raw.githubusercontent.com/sclaudiobr/geodb/main/saneamento_restricoes_vs14.sql

from pathlib import Path
arquivo_sql = Path("/content") / "saneamento_restricoes_vs14.sql"
print("Arquivo baixado:", arquivo_sql)

--2026-07-22 22:07:05--  https://raw.githubusercontent.com/sclaudiobr/geodb/main/saneamento_restricoes_vs14.sql
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8519 (8.3K) [text/plain]
Saving to: ‘/content/saneamento_restricoes_vs14.sql’

/content/saneamento 100%[===================>]   8.32K  --.-KB/s    in 0s      

2026-07-22 22:07:06 (71.7 MB/s) - ‘/content/saneamento_restricoes_vs14.sql’ saved [8519/8519]

Arquivo baixado: /content/saneamento_restricoes_vs14.sql


## 4. Importar o arquivo SQL — professor executa


In [ ]:
resultado = subprocess.run(
    [
        "sudo", "-u", "postgres",
        "psql",
        "--set", "ON_ERROR_STOP=on",
        "--dbname", "postgres",  # Conecta no banco neutro aqui
        "--file", str(arquivo_sql)
    ],
    text=True,
    capture_output=True
)

if resultado.returncode != 0:
    print("--- STDOUT ---")
    print(resultado.stdout)
    print("--- STDERR ---")
    print(resultado.stderr)
    raise RuntimeError("Falha ao importar o banco.")

print("Banco importado com sucesso.")
if resultado.stdout:
    print(resultado.stdout[-1500:])


Banco importado com sucesso.
 pg_terminate_backend 
----------------------
(0 rows)

DROP DATABASE
CREATE DATABASE
You are now connected to database "Aula9" as user "postgres".
CREATE SCHEMA
CREATE EXTENSION
CREATE EXTENSION
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE TABLE
CREATE SEQUENCE
ALTER SEQUENCE
CREATE TABLE
CREATE SEQUENCE
ALTER SEQUENCE
CREATE TABLE
CREATE SEQUENCE
ALTER SEQUENCE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
ALTER TABLE
ALTER TABLE
CREATE TRIGGER
CREATE TRIGGER
CREATE TRIGGER
 setval 
--------
      7
(1 row)

 setval 
--------
     52
(1 row)

 setval 
--------
     28
(1 row)




## 5. Criar a conexão do Python com o banco

A variável `engine` representa a conexão que será utilizada pelo Pandas e
pelo GeoPandas.


In [ ]:
from urllib.parse import quote_plus

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import subprocess

DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{quote_plus(DB_PASSWORD)}@"
    f"localhost:5432/{DB_NAME}"
)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)
print("Conexão criada.")

# ... [Seu código intermediário que usa o engine] ...

# ANTES DE RODAR O SUBPROCESS: Fecha todas as conexões e limpa o pool
engine.dispose()
print("Pool de conexões limpo. O banco foi liberado.")

# Agora o subprocess consegue rodar o DROP DATABASE sem ser bloqueado
resultado = subprocess.run(
    [
        "sudo", "-u", "postgres",
        "psql",
        "--set", "ON_ERROR_STOP=on",
        "--dbname", "postgres", # Conectando no banco neutro aqui no psql
        "--file", str(arquivo_sql)
    ],
    text=True,
    capture_output=True
)


Conexão criada.
Pool de conexões limpo. O banco foi liberado.


## 6. Testar PostgreSQL e PostGIS


In [ ]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import subprocess

# 1. Defina a conexão apontando para o banco padrão 'postgres'
# Isso evita o erro de "banco já existe" ou "banco em uso" no passo de remoção
DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{quote_plus(DB_PASSWORD)}@"
    f"localhost:5432/Aula9"
)

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True
)

print("Conexão criada com o banco administrativo.")

# 2. Executa a checagem de versões
with engine.connect() as conexao:
    versao_pg = conexao.execute(
        text("SELECT version();")
    ).scalar()

    # Nota: PostGIS geralmente é instalado por banco.
    # Se falhar aqui por não estar no banco 'postgres', comente esta linha.
    try:
        versao_postgis = conexao.execute(
            text("SELECT PostGIS_Full_Version();")
        ).scalar()
    except Exception:
        versao_postgis = "PostGIS não instalado no banco padrão 'postgres'."

print("PostgreSQL:")
print(versao_pg)
print("\nPostGIS:")
print(versao_postgis)

# 3. Fecha explicitamente todas as conexões abertas do pool
engine.dispose()
print("\nPool de conexões liberado.")

# 4. Executa a importação do arquivo estrutural via psql
resultado = subprocess.run(
    [
        "sudo", "-u", "postgres",
        "psql",
        "--set", "ON_ERROR_STOP=on",
        "--dbname", "postgres",  # Mantém a conexão no banco neutro
        "--file", str(arquivo_sql)
    ],
    text=True,
    capture_output=True
)

if resultado.returncode != 0:
    print("\n--- ERRO NA IMPORTAÇÃO ---")
    print(resultado.stderr)
    raise RuntimeError("Falha ao importar o banco.")

print("\nBanco importado com sucesso.")


Conexão criada com o banco administrativo.
PostgreSQL:
PostgreSQL 14.23 (Ubuntu 14.23-0ubuntu0.22.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0, 64-bit

PostGIS:
POSTGIS="3.4.1 ca035b9" [EXTENSION] PGSQL="140" GEOS="3.12.1-CAPI-1.18.1" PROJ="9.3.1 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=/tmp/proj DATABASE_PATH=/usr/share/proj/proj.db" LIBXML="2.9.13" LIBJSON="0.15" LIBPROTOBUF="1.3.3" WAGYU="0.5.0 (Internal)" TOPOLOGY

Pool de conexões liberado.

Banco importado com sucesso.


## 7. Importar camadas de dados geoespaciais

In [ ]:
!pip install geopandas shapely psycopg2-binary

In [ ]:
# URLs para download via wget
urls_wget = [
    "https://raw.githubusercontent.com/sclaudiobr/geodb/main/ocorrencia.geojson",
    "https://raw.githubusercontent.com/sclaudiobr/geodb/main/trecho.geojson",
    "https://raw.githubusercontent.com/sclaudiobr/geodb/main/bairro.geojson"
]

# Baixar todos com wget
for url in urls_wget:
    nome = url.split('/')[-1]
    !wget -O /content/$nome $url

print("\n✅ Downloads concluídos!")

# Listar os arquivos baixados
from pathlib import Path
arquivos = list(Path("/content").glob("*.geojson"))
print(f"📁 Arquivos baixados: {len(arquivos)}")
for arq in arquivos:
    print(f"   {arq.name}: {arq.stat().st_size / 1024:.2f} KB")

In [ ]:
import geopandas as gpd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# 1. Conexão com o banco final 'Aula9'
DATABASE_URL_AULA9 = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{quote_plus(DB_PASSWORD)}@"
    f"localhost:5432/Aula9"
)

engine_aula9 = create_engine(DATABASE_URL_AULA9)

caminho_geojson = "/content/trecho.geojson"
nome_da_tabela = "trecho"

# 2. DROP FORÇADO: Remove a tabela e qualquer dependência (Views, Triggers, Constraints)
print(f"Executando DROP forçado na tabela '{nome_da_tabela}' se ela existir...")
with engine_aula9.connect() as conexao:
    # O CASCADE força a exclusão mesmo se houver objetos vinculados à tabela
    conexao.execute(text(f'DROP TABLE IF EXISTS "{nome_da_tabela}" CASCADE;'))
    # Garante a gravação imediata da exclusão no banco de dados
    conexao.commit()

# 3. Leitura e importação do GeoJSON
print(f"Lendo o arquivo GeoJSON: {caminho_geojson}...")
gdf = gpd.read_file(caminho_geojson)

print(f"Enviando {len(gdf)} registros para a nova tabela '{nome_da_tabela}'...")

# 4. Salva os novos dados geográficos
# Usamos 'append' aqui porque o DROP manual já limpou o espaço de forma forçada
# 4. Salva os novos dados geográficos (a coluna geométrica é detectada automaticamente)
gdf.to_postgis(
    name=nome_da_tabela,
    con=engine_aula9,
    if_exists="append",
    index=False
)

print("Camada substituída com sucesso via DROP forçado!")
engine_aula9.dispose()


Executando DROP forçado na tabela 'trecho' se ela existir...
Lendo o arquivo GeoJSON: /content/trecho.geojson...
Enviando 6 registros para a nova tabela 'trecho'...
Camada substituída com sucesso via DROP forçado!


##8. Listar as tabelas

Aqui o Python pergunta ao banco quais tabelas estão disponíveis.


In [ ]:
consulta_tabelas = """
SELECT
    table_schema AS esquema,
    table_name AS tabela,
    table_type AS tipo
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tabelas = pd.read_sql_query(consulta_tabelas, engine)
tabelas


,esquema,tabela,tipo
0,public,bairro,BASE TABLE
1,public,geography_columns,VIEW
2,public,geometry_columns,VIEW
3,public,ocorrencia,BASE TABLE
4,public,spatial_ref_sys,BASE TABLE
5,public,trecho,BASE TABLE


## 9. Primeira consulta SQL

- `SELECT`: escolhe as colunas;
- `FROM`: indica a tabela;
- `ORDER BY`: organiza o resultado;
- `LIMIT`: limita a quantidade de registros.


In [ ]:
consulta = """
SELECT
    nome,pop_2010,pop_2018,area_m2
FROM bairro
ORDER BY nome DESC
LIMIT 5;
"""

pd.read_sql_query(consulta, engine)

,nome,pop_2010,pop_2018,area_m2
0,SÃO GERALDO,7599,5196,1045002
1,PRESIDENTE VARGAS,7944,9945,567041
2,PRAÇA 14 DE JANEIRO,10250,11755,1003363
3,NOSSA SENHORA DAS GRACAS,15116,11368,2116989
4,CENTRO,33183,25738,4269417


In [ ]:
consulta = """
SELECT
    id_trecho,material,extensao
FROM trecho
ORDER BY id_trecho DESC
LIMIT 5;
"""
pd.read_sql_query(consulta, engine)

,id_trecho,material,extensao
0,39,ACO,1187.76
1,37,MAG,211.31
2,36,CIM,1055.43
3,35,PVC,229.76
4,34,PVA,515.08


## 10. Pequena interação: alterar um filtro

Troque o valor de `zona_escolhida` e execute novamente.

Opções: `Sul`, `Centro-Sul`, `Norte` e `Oeste`.


In [ ]:
zona_escolhida = "Oeste"

consulta_zona = text("""
SELECT
    nome,
    zona,
    populacao_estimada
FROM bairros_demo
WHERE zona = :zona
ORDER BY populacao_estimada DESC;
""")

pd.read_sql_query(
    consulta_zona,
    engine,
    params={"zona": zona_escolhida}
)


## 10. Carregar geometrias com GeoPandas

O GeoPandas recebe a coluna `geom` e transforma o resultado em um
`GeoDataFrame`.


In [ ]:
consulta_bairros = """
SELECT
    id,
    nome,
    zona,
    populacao_estimada,
    area_km2,
    geom
FROM bairros_demo
ORDER BY id;
"""

bairros = gpd.read_postgis(
    consulta_bairros,
    con=engine,
    geom_col="geom"
)

print("Quantidade de bairros:", len(bairros))
print("Sistema de referência:", bairros.crs)
bairros.head()


ProgrammingError: (psycopg2.errors.SyntaxError) syntax error at or near "("
LINE 5:         ST_Length(ST_Transform(geometry, 31982)) AS comprime...
                         ^

[SQL: 
    SELECT 
        id_trecho,
        tipo
        ST_Length(ST_Transform(geometry, 31982)) AS comprimento_metros,
        geometry
    FROM "trecho";
]
(Background on this error at: https://sqlalche.me/e/20/f405)

## 11. Gerar um mapa temático

A população estimada será usada para diferenciar os polígonos.
Os números e limites são fictícios.


In [ ]:
ax = bairros.plot(
    column="populacao_estimada",
    legend=True,
    figsize=(10, 8)
)

ax.set_title("População estimada — dados didáticos fictícios")
ax.set_axis_off()
plt.show()


## 12. Visualizar pontos de interesse


In [ ]:
equipamentos = gpd.read_postgis(
    """
    SELECT
        id,
        nome,
        categoria,
        capacidade,
        geom
    FROM equipamentos_publicos_demo
    ORDER BY id;
    """,
    con=engine,
    geom_col="geom"
)

equipamentos.head()


In [ ]:
ax = bairros.plot(
    figsize=(10, 8),
    alpha=0.35
)

equipamentos.plot(
    ax=ax,
    markersize=45
)

ax.set_title("Bairros e equipamentos públicos fictícios")
ax.set_axis_off()
plt.show()


## 13. Consulta espacial: equipamentos dentro de cada bairro

`ST_Contains` verifica se um ponto está dentro de um polígono.


In [ ]:
consulta_espacial = """
SELECT
    b.nome AS bairro,
    b.zona,
    COUNT(e.id) AS quantidade_equipamentos
FROM bairros_demo AS b
LEFT JOIN equipamentos_publicos_demo AS e
    ON ST_Contains(b.geom, e.geom)
GROUP BY b.id, b.nome, b.zona
ORDER BY quantidade_equipamentos DESC, b.nome;
"""

pd.read_sql_query(consulta_espacial, engine)


## 14. Consulta espacial por distância

A consulta procura equipamentos em um raio definido a partir de um ponto.
O aluno pode alterar somente o valor de `raio_km`.


In [ ]:
longitude_referencia = -60.0217
latitude_referencia = -3.1190
raio_km = 5

consulta_distancia = text("""
SELECT
    nome,
    categoria,
    ROUND(
        ST_Distance(
            geom::geography,
            ST_SetSRID(
                ST_MakePoint(:longitude, :latitude),
                4326
            )::geography
        )::numeric / 1000,
        2
    ) AS distancia_km
FROM equipamentos_publicos_demo
WHERE ST_DWithin(
    geom::geography,
    ST_SetSRID(
        ST_MakePoint(:longitude, :latitude),
        4326
    )::geography,
    :raio_metros
)
ORDER BY distancia_km;
""")

resultado_distancia = pd.read_sql_query(
    consulta_distancia,
    engine,
    params={
        "longitude": longitude_referencia,
        "latitude": latitude_referencia,
        "raio_metros": raio_km * 1000
    }
)

resultado_distancia


## 15. Mapa interativo com Folium


In [ ]:
bairros_4326 = bairros.to_crs(epsg=4326)
equipamentos_4326 = equipamentos.to_crs(epsg=4326)

mapa = folium.Map(
    location=[-3.09, -60.03],
    zoom_start=11,
    control_scale=True
)

folium.GeoJson(
    bairros_4326.to_json(),
    name="Bairros",
    tooltip=folium.GeoJsonTooltip(
        fields=["nome", "zona", "populacao_estimada"],
        aliases=["Bairro:", "Zona:", "População estimada:"]
    )
).add_to(mapa)

for _, registro in equipamentos_4326.iterrows():
    folium.Marker(
        location=[
            registro.geometry.y,
            registro.geometry.x
        ],
        tooltip=registro["nome"],
        popup=(
            f"<b>{registro['nome']}</b><br>"
            f"Categoria: {registro['categoria']}<br>"
            f"Capacidade: {registro['capacidade']}"
        )
    ).add_to(mapa)

folium.LayerControl().add_to(mapa)
mapa


## 16. Atividade rápida de interpretação

1. Qual tabela guarda os polígonos?
2. Qual tabela guarda os pontos?
3. O que mudou quando a zona foi alterada?
4. Qual bairro possui mais equipamentos na demonstração?
5. O que acontece ao aumentar ou diminuir `raio_km`?
6. Qual é a função da coluna `geom`?
7. Qual foi o papel do SQL?
8. Qual foi o papel do Python?

**Mensagem principal:** o banco armazena e consulta; o Python organiza,
analisa e apresenta; o PostGIS entende as geometrias.


## Observação final

O ambiente e o banco criados no Colab são temporários. Caso a sessão seja
reiniciada, execute novamente as etapas de preparação e importação.
